# sample 1

In [73]:


from anthropic import BaseModel
from httpcore import default_ssl_context
# 1、モデルの初期化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print

# .envファイルから環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [74]:
from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物情報"""
    name: str = Field(description="氏名")
    age: int = Field(description="年齢")
    occupation: str = Field(description="職業")


# 構造化出力の LLM を作成
structured_llm = model.with_structured_output(Person)
# 呼び出し
result = structured_llm.invoke("太郎さんは30歳のソフトウェアエンジニアです")
print(result)
print(type(result))
# result は Person のインスタンス
print(result.name)  # "太郎"
print(result.age)  # 30
print(result.occupation)  # "ソフトウェアエンジニア"

Person(name='太郎', age=30, occupation='ソフトウェアエンジニア')

<class '__main__.Person'>

太郎

30

ソフトウェアエンジニア

# sample 2

In [75]:
class Movie(BaseModel):
    """映画の詳細情報"""
    title: str = Field(description="映画タイトル")
    year: int = Field(description="公開年")
    director: str = Field(description="監督")
    rating: float = Field(description="映画評価、10点満点")


structured_model = model.with_structured_output(Movie)
result = structured_model.invoke("映画『インセプション』の情報を教えてください。")
print(result)

Movie(title='インセプション', year=2010, director='クリストファー・ノーラン', rating=8.8)

# sample 3

In [76]:
from pydantic import BaseModel, Field


# 出力構造を定義
class SentimentAnalysis(BaseModel):
    """感情分析結果"""
    sentiment: str = Field(description="感情の傾向：positive/negative/neutral")
    confidence: float = Field(description="信頼度、0〜1の間")
    keywords: list[str] = Field(description="キーワードリスト")


# ✅ v1.x：with_structured_output を使用
structured_model = model.with_structured_output(SentimentAnalysis)
# 呼び出し
text = "このコースの内容はとても実用的で、多くの知識を学べました。強くお勧めします！"
result = structured_model.invoke(
    f"以下のテキストの感情を分析してください：\n{text}"
)
print(f"型: {type(result)}")  # <class 'SentimentAnalysis'>
print(f"感情: {result.sentiment}")
print(f"信頼度: {result.confidence}")
print(f"キーワード: {result.keywords}")


型: <class '__main__.SentimentAnalysis'>

感情: positive

信頼度: 0.95

キーワード: ['実用的', '知識', '学べました', '強くお勧め']

# 使用可能なフィールド

In [77]:
class Person(BaseModel):
    """人物情報"""
    name: str = Field(description="氏名")
    age: int = Field(description="年齢")
    occupation: str = Field(description="職業")


# 構造化出力の LLM を作成
structured_llm = model.with_structured_output(Person)
# 呼び出し
result = structured_llm.invoke("太郎さんはソフトウェアエンジニアです")
print(result)

Person(name='太郎', age=30, occupation='ソフトウェアエンジニア')

## 比較

In [78]:
from typing import Optional


class Person(BaseModel):
    """人物情報"""
    name: str = Field(description="氏名")
    age: Optional[int] = Field(description="年齢")
    occupation: str = Field(description="職業")


# 構造化出力の LLM を作成
structured_llm = model.with_structured_output(Person)
# 呼び出し
result = structured_llm.invoke("太郎さんはソフトウェアエンジニアです")
print(result)

Person(name='太郎', age=None, occupation='ソフトウェアエンジニア')

### 2.2 デフォルト値
異なるモデルプロバイダーでは、このフィールドのサポート状況が異なります

In [79]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model_with_openrouter = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


class Person(BaseModel):
    """人物情報"""
    name: str = Field(description="氏名")
    age: int = Field(default=10, description="年齢")
    occupation: str = Field(description="職業")


# 構造化出力の LLM を作成
structured_llm = model.with_structured_output(Person)
# 呼び出し
result = structured_llm.invoke("太郎さんはソフトウェアエンジニアです")
print(result)

Person(name='太郎', age=30, occupation='ソフトウェアエンジニア')

In [80]:
class Config(BaseModel):
    timeout: Optional[int] = Field(30, description="タイムアウト時間（秒単位）")
    retry: bool = Field(False, description="リトライをサポートするかどうか")
    max_attempts: int = Field(6, description="最大リトライ回数")


# 構造化出力の LLM を作成
structured_llm = model.with_structured_output(Config)
# 呼び出し
result = structured_llm.invoke("設定要件：リトライをサポートし、最大5回まで再試行する")
print(result)

Config(timeout=None, retry=True, max_attempts=5)

In [81]:
from enum import Enum


class Proiority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"


class Task(BaseModel):
    title: str
    priority: Proiority


class CustomerInfo(BaseModel):
    "顧客情報"
    name: str = Field(description="顧客氏名")
    phone: str = Field(description="電話番号")
    email: Optional[str] = Field(description="メールアドレス")
    issue: str = Field(description="問題の説明")
    urgency: Proiority = Field(description="緊急度")
    # テスト


structured_llm = model_with_openrouter.with_structured_output(CustomerInfo)
conversation = """
カスタマーサポート: こんにちは、どのようなご用件でしょうか？
顧客: 私は田中太郎です、電話番号 090-1234-5678、注文した商品がずっと発送されず、とても困っています！
カスタマーサポート: かしこまりました、確認いたします
"""
result = structured_llm.invoke(f"以下のカスタマーサポート対話から顧客情報を抽出してください：\n{conversation}")
print(result)
print("\n抽出結果：")
print(f" 顧客: {result.name}")
print(f" 電話: {result.phone}")
print(f" メール: {result.email or '未提供'}")
print(f" 問題: {result.issue}")
print(f" 緊急度: {result.urgency.value}")

CustomerInfo(
    name='田中太郎',
    phone='090-1234-5678',
    email=None,
    issue='注文した商品がずっと発送されず、とても困っている',
    urgency=<Proiority.HIGH: '高'>
)

抽出結果：

顧客: 田中太郎

電話: 090-1234-5678

メール: 未提供

問題: 注文した商品がずっと発送されず、とても困っている

緊急度: 高

In [82]:

from typing import Literal
from enum import Enum


class CustomerInfo(BaseModel):
    "顧客情報"
    name: str = Field(description="顧客氏名")
    phone: str = Field(description="電話番号")
    email: Optional[str] = Field(description="メールアドレス")
    issue: str = Field(description="問題の説明")
    urgency: Literal["低", "中", "高"] = Field(description="緊急度")
    # テスト


structured_llm = model_with_openrouter.with_structured_output(CustomerInfo)
conversation = """
カスタマーサポート: こんにちは、どのようなご用件でしょうか？
顧客: 私は田中太郎です、電話番号 090-1234-5678、注文した商品がずっと発送されず、とても困っています！
カスタマーサポート: かしこまりました、確認いたします
"""
result = structured_llm.invoke(f"以下のカスタマーサポート対話から顧客情報を抽出してください：\n{conversation}")
print(result)
# print("\n抽出結果：")
# print(f" 顧客: {result.name}")
# print(f" 電話: {result.phone}")
# print(f" メール: {result.email or '未提供'}")
# print(f" 問題: {result.issue}")
# print(f" 緊急度: {result.urgency.value}")


CustomerInfo(
    name='田中太郎',
    phone='090-1234-5678',
    email=None,
    issue='注文した商品がずっと発送されず、とても困っている',
    urgency='高'
)

In [83]:
from typing import List


class Person(BaseModel):
    """人物情報"""
    name: str = Field(description="氏名")
    age: int = Field(description="年齢")


class PersonList(BaseModel):
    """人物リスト"""
    people: List[Person]  # 複数の Person オブジェクト


structured_model = model.with_structured_output(PersonList, method="function_calling")

result = structured_model.invoke("太郎さん30歳、次郎さん40歳")
print(result)

PersonList(people=[Person(name='太郎', age=30), Person(name='次郎', age=40)])

## 例2

In [84]:
class Review(BaseModel):
    """製品レビュー"""
    product: str
    rating: int = Field(description="評価 1〜5点")
    pros: List[str] = Field(description="長所のリスト")
    cons: List[str] = Field(description="短所のリスト")


structured_model = model.with_structured_output(Review, method="function_calling")

result = structured_model.invoke("iPhone 17は素晴らしい！カメラが強力で、持った感じも良い。ただし価格が高く、充電器が付属していない。4点。")
print(result)

Review(
    product='iPhone 17',
    rating=4,
    pros=['カメラが強力', '持った感じが良い'],
    cons=['価格が高い', '充電器が付属していない']
)

例3

In [85]:
class Invoice(BaseModel):
    """請求書情報"""
    invoice_number: str = Field(description="請求書番号")
    date: str = Field(description="日付")
    total_amount: float = Field(description="合計金額")
    items: List[str] = Field(description="商品")


# テスト
structured_llm = model.with_structured_output(Invoice, method="function_calling")

invoice_text = """
請求書番号: INV-2024-001
日付: 2024-01-15
合計金額: 1299.00
商品: MacBook Pro, AppleCare+
"""

invoice = structured_llm.invoke(f"請求書情報を抽出してください：{invoice_text}")
print(invoice)

Invoice(invoice_number='INV-2024-001', date='2024-01-15', total_amount=1299.0, items=['MacBook Pro', 'AppleCare+'])

ネスト構造

In [86]:
class Address(BaseModel):
    """場所の説明"""
    city: str = Field(description="都市")
    district: str = Field(description="エリア")


class Company(BaseModel):
    """会社情報"""
    name: str = Field(description="会社名")
    address: Address = Field(description="会社の所在地")


structured_model = model.with_structured_output(Company, method="function_calling")
result = structured_model.invoke("アリババは杭州の浜江区にあります")
print(result)

Company(name='アリババ', address=Address(city='杭州市', district='浜江区'))

In [87]:
from pydantic import BaseModel, Field
from typing import List


# 1. ネストされた Pydantic モデルを定義
class Actor(BaseModel):
    """俳優情報"""
    name: str = Field(description="俳優名")
    role: str = Field(description="演じる役柄")


class Movie(BaseModel):
    """映画情報"""
    title: str = Field(description="映画タイトル")
    year: int = Field(description="公開年")
    director: str = Field(description="監督")
    cast: List[Actor] = Field(description="俳優リスト")  # リストフィールドを定義
    rating: float = Field(description="評価")


# 2. モデルを初期化して出力構造をバインド
structured_model = model.with_structured_output(Movie, method="function_calling")
# 3. モデルを呼び出し、直接 Movie インスタンスを取得
response = structured_model.invoke("映画『インセプション』を紹介してください")
print(response)

Movie(
    title='インセプション',
    year=2010,
    director='クリストファー・ノーラン',
    cast=[
        Actor(name='レオナルド・ディカプリオ', role='ドム・コブ'),
        Actor(name='ジョセフ・ゴードン＝レヴィット', role='アーサー'),
        Actor(name='エリオット・ページ', role='アリアドネ'),
        Actor(name='トム・ハーディ', role='Eames'),
        Actor(name='ケン・ワタナベ', role='サイード'),
        Actor(name='マリオン・コティヤール', role='モル')
    ],
    rating=8.8
)

In [88]:
from pydantic import BaseModel
from typing import List


class Aspect(BaseModel):
    """レビュー観点"""
    name: str = Field(description="観点名、例：品質、価格、サービス")
    score: int = Field(description="評価、1〜5")
    comment: str = Field(description="具体的な評価")


class ProductReview(BaseModel):
    """製品レビュー分析"""
    overall_sentiment: str = Field(description="全体の感情：positive/negative/neutral")
    overall_score: int = Field(description="総合評価、1〜5")
    aspects: List[Aspect] = Field(description="各観点の評価")
    summary: str = Field(description="一言まとめ")


# 構造化モデルを作成
structured_model = model.with_structured_output(ProductReview, method="function_calling")
# テスト
review_text = """
このノートパソコンの性能は非常に強力で、大型ソフトウェアも快適に動作します。
画面の色彩が鮮やかで、動画視聴も快適です。
ただし価格が少し高く、ファンの音がやや大きいです。
カスタマーサポートの対応が良く、配送も速いです。
総じて購入する価値はあると思います。
"""
result = structured_model.invoke(
    f"以下の製品レビューを分析してください：\n{review_text}"
)
print(f"全体の感情: {result.overall_sentiment}")
print(f"総合評価: {result.overall_score}/5")
print(f"\n各観点の評価:")
for aspect in result.aspects:
    print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\nまとめ: {result.summary}")


全体の感情: positive

総合評価: 4/5

各観点の評価:

- 性能: 5/5 - 非常に強力で、大型ソフトウェアも快適に動作します。

- 画面: 5/5 - 画面の色彩が鮮やかで、動画視聴も快適です。

- 価格: 3/5 - 価格が少し高いです。

- ファンの音: 3/5 - ファンの音がやや大きいです。

- カスタマーサポート: 5/5 - カスタマーサポートの対応が良く、配送も速いです。

まとめ: 総じて購入する価値はあると思います。

制約条件

In [89]:
from pydantic import ValidationError

class User(BaseModel):
    name : str = Field(description="氏名",min_length=2,max_length=50)
    age : int = Field(description="年齢",le=150)
    email : str = Field(description="メールアドレス")


try:
    user1 = User(name="tom",age = 20,email="tom@126.com")
    print(f"[OK]{user1}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[OK]name='tom' age=20 email='tom@126.com'

エラー

In [90]:
from pydantic import ValidationError

try:
    user2 = User(name="tom",age = 200,email="tom@126.com")
    print(f"[OK]{user2}")
except ValidationError as e:
    print(f"[FAIL]{e}")

[FAIL]1 validation error for User
age
  Input should be less than or equal to 150 
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal

In [91]:
class Product(BaseModel):
    """製品情報（厳格な検証）"""
    name: str = Field(description="製品名（文字列型）", min_length=2)
    price: float = Field(description="価格、数値型", gt=0)
    stock: int = Field(description="在庫、整数型", ge=0)
# テスト
structured_llm = model.with_structured_output(Product)
# response = structured_llm.invoke("Huawei Mate 80 Pro Maxの価格は7999、現在の在庫は100")
response = structured_llm.invoke("Huawei Mate 80 Pro Maxの価格は-7999、現在の在庫は-100")
print(response)

Product(name='Huawei Mate 80 Pro Max', price=7999.0, stock=100)